In [65]:
import requests
import json
import time
import os

def text_to_speech(text, verbose=True):
    def request_synthesis(text):
        base_url = "http://127.0.0.1:7860"
        endpoint = "/queue/join"
        payload = json.dumps({
            "data": ["EN-US", text, 1, "EN"],
            "event_data": None,
            "fn_index": 1,
            "trigger_id": 8,
            "session_hash": "60236rt7l1u"
        })
        
        url = base_url + endpoint
        req = requests.post(url, payload)
        return req.json()

    def get_folder(url="http://127.0.0.1:5000", max_retries=20, delay=0.5, first=False):
        if first:
            return requests.get(url).json()['latest_folder']
        return requests.get(url).json()['latest_folder']

    def get_audio(folder_name, max_retries=20, delay=0.5):
        url = f"http://127.0.0.1:7860/file=/tmp/gradio/{folder_name}/audio"
        req = requests.get(url)
        return req.content

    def to_file(audio, file_name="temp"):
        audio_file = f"{file_name}.wav"
        with open(audio_file, "wb") as f:
            f.write(audio)
        return audio_file

    def play(audio_file):
        import sounddevice as sd
        import soundfile as sf
        data, fs = sf.read(audio_file)
        sd.play(data, fs)
        sd.wait()

    try:
        initial_folder = get_folder(first=True)
        if verbose: print(f"Initial folder: {initial_folder}")

        job_hash = request_synthesis(text)
        if verbose: print(f"Job hash: {job_hash}")
        
        # Wait for a new folder
        new_folder = initial_folder
        max_retries = 30
        for _ in range(max_retries):
            time.sleep(1)
            new_folder = get_folder()
            if new_folder != initial_folder:
                break
        
        if new_folder == initial_folder:
            raise Exception("No new folder detected after maximum retries")
        
        if verbose: print(f"New folder detected: {new_folder}")
        
        audio = get_audio(new_folder)
        audio_file = to_file(audio=audio)
        play(audio_file)
        return "success"
    except Exception as e:
        print(f"Error: {e}")
        return "failed"

# Example usage
para = \
"""
The sun was setting over the bustling streets of Tokyo, casting a warm orange glow over the crowded sidewalks. The
sound of laughter and chatter filled the air as people made their way home from work or out for dinner with
friends. In the midst of this vibrant scene, a small group of street performers had gathered on the corner of a
quiet side street, drawing in a crowd with their lively music and acrobatic tricks. A young musician, perched on
top of a stack of boxes, strummed the strings of his guitar with infectious energy, while a pair of jugglers
expertly kept three glowing balls aloft as they weaved through the gathering crowd. As the sun dipped below the
horizon, the performers paused to take a bow, their faces flushed with excitement and admiration from the
delighted onlookers.
"""
result = text_to_speech(para)
print(result)

Initial folder: 81319c5da276e7a7084a561274ceec3efabb787815f0d2033fa3ef697d051c6e
Job hash: {'event_id': '359072dc2506489ab9ada85cbca828b9'}
New folder detected: b0e52c62a64968b3122eb776a3705fafd520c9b9b70fc89156c9d79435d410fb
success


In [21]:
%pip install soundfile

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 3.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [58]:
import threading 
import sounddevice as sd

url = "http://127.0.0.1:5000"

req = requests.get(url)

req.content

b'{"latest_folder": "No folder detected yet"}\n'

In [59]:
req.json()

{'latest_folder': 'No folder detected yet'}

In [13]:
folder = req.content.decode('utf-8').strip()
folder

'{"latest_folder": "50d7a7dd75e1b4505ec93f9dc6479afb1453df546bcb46e62ff66efb6720726b"}'

In [18]:
folder_name = folder.split('"')[3]
folder_name

'50d7a7dd75e1b4505ec93f9dc6479afb1453df546bcb46e62ff66efb6720726b'

In [19]:
req = requests.get(f"http://127.0.0.1:7860/file=/tmp/gradio/{folder_name}/audio")
req.content

b'RIFF^Y\x06\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00D\xac\x00\x00\x88X\x01\x00\x02\x00\x10\x00data:Y\x06\x00\xff\xff\xff\xff\x00\x00\xff\xff\xff\xff\xff\xff\xff\xff\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\x00\x00\x00\x00\x00\x00\xff\xff\xff\xff\xff\xff\x00\x00\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\x00\x00\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff

In [20]:
with open('test.wav', 'wb') as f:
    f.write(req.content)

In [1]:
from faster_whisper import WhisperModel


whisper = WhisperModel("base", device="cuda", compute_type="float16")
transcribtions = whisper.transcribe('temp.wav', beam_size=5)
print(transcribtions)

C:\Users\dasha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 